## Chapter 02 

## RADONIKA LIBRARY

We start building the library now. We will have the standard structure. We will come back to this later once the library is prepared.

## 2.1 PROCESS LAYER

---

The file `process.py` is a 2-D stochastic process engine. Instead of reading it from line 1 to the end, we will build it bottom-up. Starting with a concrete process, finding what logic repeats, extracting that logic into base classes and helpers, and gradually expanding into more complex math. 

Every object in `process.py` models a 2-D trajectory $(X_t, Y_t)$. Rather than writing separate simulation, density, and plotting code for every single random process, we will make a file that has a separate definition for things that we will use again and again.

```text
Concrete Processes (normal, bm, poisson, stocks...)
├── Defined using their unique maths, how to step forward
│
└── Intermediate Wrappers (_IID)
    ├── Groups processes that are iid
    │
    └── Class (Process)
        ├── Provides the universal engine: paths(), sample(), pdf(), measure tilting, & UI plots
        │
        └── 4 tiny helpers (_pair, _vol, _bvn_pdf, _gauss_paths)
```

## 2.1.1 Two-Dimensional iid Processes

Since we plan on making $iid$ Cauchy, Normal, Uniform etc, the main underlying engine of generating the array etc can be created separately under something along the lines of a engine that draws randomly to make a $iid$ process. We can name it `iid` ofcourse. Hence, they will all form a subclass as `class normal(iid)`, `class cauchy(iid)`, etc. Then the `class iid` can have the mechanisms to generate the process and look at distributions at several time points etc. This stucture is fine but there are more technicalities. In this library, we are going to have processes that are non-$iid$s too. We will want to simulate these paths too and also look at the distributions at several time points. What now? 

There is an easy fix for this. Make `iid` itself a subclass of an even more fundamental class named `Process`. This will have all forms of the functions we want to do to our processes. It will have methods such as `simulate` that will be called again and again and defined as per our need. This flexibility is needed since the way to simulate $iid$ processes is obviously different from the way to simulate Brownian motion.

In [1]:
class normal:
    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")                  #the _pair helper is used
        super().__init__([stats.norm(m[k], s[k]) for k in range(2)], seed,
                         f"normal(mean={fmt(m)}, sigma={fmt(s)})")

Due to some technicalities, we will use `_IID` instead of `iid` (we will see why later). More precisely, above code should then be `normal(_IID):` where `_IID` will simulate the trajectories. That will further need `Process` as we said before. So, let us start there now, from making the `class Process`.

The general structure (as we are imagining now) is :
```text
Process
(will have some fundamental methods that all processes in this file will have)
    ├── "_IID(Process)"
    │   (fundamental methods but only those special to iid processes)
    │       ├── "normal"
    │       ├── "cauchy"
    │       └── "uniform", etc.
    │
    ├── "non-iid things 1"
    │   (methods special to this process)
    │
    └── "non-iid things 2"
        (methods special to this process)
        etc.  
```

We can start building the actual thing now. The most important thing for us is the overall shape of these 2D Stochastic processes in $\mathbb{R}^3$.
```text
shape = (n, len(times), 2)
         │      │        └─ the two coordinates, X and Y
         │      └────────── how many time points
         └───────────────── how many paths
```
If we had to do this with nothing else, the whole Process would be this:

In [2]:
class Process:
    def simulate(self, n, times, start, rng):
        raise NotImplementedError

`Process` itself has no idea how to simulate anything. It says “every process must have a `simulate`” and raises `NotImplementedError` if a subclass forgets. Each specific process (Cauchy, Brownian motion, and so on) fills in the actual recipe. We will now make all these small building blocks for the `class Process` such as `_rng`, `simulate`, `_start`, `paths` etc. We have already discussed what `rng` does in Python. Let us see what the others are doing one by one.

One rule is written in the docstring that `times[0]` must be 0, and the first column of the output must equal `start`. Every path starts at the starting point. The `simulate` also needs `start`, the point `(x0, y0)` where every path begins. The user might give it in a few different ways, or not at all, so the process needs one place that turns whatever they gave into a clean array.

In [3]:
def _start(self, start):
    s = self.default_start if start is None else start
    s = np.asarray(s, float)
    if s.ndim == 0:
        s = np.repeat(s, self.dim)
    if s.shape != (self.dim,):
        raise ValueError(f"start must have {self.dim} coordinates")
    return s

It does four small jobs, in order:

1. Fill in a default. If the user passed nothing (None), use `self.default_start`, which is `(0.0, 0.0)` for most processes. Stock prices override it to `(1.0, 1.0)`, because a price starting at 0 would be stuck at 0 forever(we will see why when we get there).
2. Convert to a NumPy array of floats. A tuple `(0, 0)` becomes array `([0., 0.])`.
3. Allow a shortcut. If the user passed a single number like 5, `s.ndim == 0` is true, and `np.repeat(s, 2)` turns it into `[5., 5.]`. So `start=5` means “start both coordinates at 5”.
4. Check the shape. After all that, the result must be exactly length 2. If not, it raises a clear error instead of failing mysteriously later.

We now have `_rng` and `_start`. We also have `simulate`. But calling `simulate` directly is clumsy because the user would have to build `times`, `start` and `rng` by hand.

`times = np.linspace(0, 1, 201)`

`X = process.simulate(20, times, process._start(None), process._rng(None))`

So `Process` gives a friendly method that does all of that named `paths`.

In [4]:
def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
    times = np.linspace(0.0, T, steps + 1)
    return times, self.simulate(n, times, self._start(start), self._rng(seed))

Note that `paths` is using `simulate` that further has `_rng` and `_start` that we built before. Every argument has a default, so `process.paths()` alone already works giving 20 paths, on $[0, 1]$, 200 steps, starting at the origin. So far we have:  

In [5]:
import numpy as np
from scipy import stats

class Process:
    """Base class: every process knows how to simulate itself and (where possible) its law."""

    name = "process"
    dim = 2
    default_start = (0.0, 0.0)

    def __init__(self, seed=None):
        self.seed = seed

    def __repr__(self):
        return self._desc

    def _start(self, start):
        """Validates or assigns default starting vector (0.0, 0.0)."""
        s = self.default_start if start is None else start
        s = np.asarray(s, float)
        if s.ndim == 0:
            s = np.repeat(s, self.dim)
        if s.shape != (self.dim,):
            raise ValueError(f"start must have {self.dim} coordinates")
        return s

    def _rng(self, seed=None):
        return np.random.default_rng(self.seed if seed is None else seed)

    def simulate(self, n, times, start, rng):
        """Paths of shape (n, len(times), dim); times[0] must be 0 and X_0 = start."""
        raise NotImplementedError

    def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
        """Simulate n paths on [0, T]; returns (times, X) with X of shape (n, steps+1, 2)."""
        times = np.linspace(0.0, T, steps + 1)
        return times, self.simulate(n, times, self._start(start), self._rng(seed))    

`paths()` gives you whole trajectories (not the plots yet ofcourse but the material that we are going to plot later are these). Often you want a different question: “at time t = 3, what values does X_t take?” To see that distribution, you need many independent draws of (X_3, Y_3), with no need for the path leading up to it. This is the law of the process at that time (similar to the 2D case). The simplest way is to simulate paths up to time t and keep only the last time point. For that we need `sample` and `sample_slices`. The cloud of points is a picture of the law. It’s dense where the process likely lands and thin where it rarely does and that is what `sample` will do. 

Where does `sample_slices` come into the picture? We will make it like an animation that shows the law evolving over time. For that we need a process to simulate the cloud at several time points and that is what `sample_slice` will do. `sample_slice` can create many clouds over many time points and `sample` just looks at them one by one.

In [6]:
def sample_slices(self, times, n, rng, start):
    X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
    return np.transpose(X[:, 1:, :], (1, 0, 2))

Recall, $X[3, 10, 1]$ means “path number 3, at the 10th time point, the Y coordinate”. So, `X[:, 1:, :]` means from the time array we are dropping the initial slice, start that array from 1. This is because we had `np.r_` that added $0$ in begining of `times`. So, something like $[1, 2, 3]$ became $[0, 1, 2, 3]$ but now we have to remove the initial time slice. The $0$ was added since that is how simulate works and the form of inputs are predefined. Once that simulation is done, we do need to return those time slices only that the user asked. The transpose here exchanges the first two component of $X$[path number, times, coordiate number] to $X$[times, path number, coordiate number]. This will read "(slice, n, 2)" which is used in next part.

In [7]:
def sample_slices(self, times, n, rng, start):
    X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
    return np.transpose(X[:, 1:, :], (1, 0, 2))
    
def sample(self, t, n=10000, start=None, seed=None):
    return self.sample_slices([t], n, self._rng(seed), self._start(start))[0]

The `[0]` at the end of the def of `sample` is simply because sample took only one time as entry, namely, `[t]` and then we want to look at them one by one. So, `[0]` says move along the $0^{th}$ coordinate (time) one by one; so basically instead of stacks of clouds, it is just one cloud at a time. Now we have the updated class as:

In [8]:
class Process:
    """Base class: every process knows how to simulate itself and (where possible) its law."""

    name = "process"
    dim = 2
    default_start = (0.0, 0.0)

    def __init__(self, seed=None):
        self.seed = seed

    def __repr__(self):
        return self._desc

    def _start(self, start):
        """Validates or assigns default starting vector (0.0, 0.0)."""
        s = self.default_start if start is None else start
        s = np.asarray(s, float)
        if s.ndim == 0:
            s = np.repeat(s, self.dim)
        if s.shape != (self.dim,):
            raise ValueError(f"start must have {self.dim} coordinates")
        return s

    def _rng(self, seed=None):
        return np.random.default_rng(self.seed if seed is None else seed)

   #----------------------------------------------------------------------------- to be provided by subclasses
    def simulate(self, n, times, start, rng):
        """Paths of shape (n, len(times), dim); times[0] must be 0 and X_0 = start."""
        raise NotImplementedError

    #this is new and we will see what it does later
    def marginal(self, t, start, k):
        """Frozen scipy distribution of coordinate k at time t (when known in closed form)."""
        raise NotImplementedError
        
    #---------------------------------------------------------------------------------------------------------

    def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
        """Simulate n paths on [0, T]; returns (times, X) with X of shape (n, steps+1, 2)."""
        times = np.linspace(0.0, T, steps + 1)
        return times, self.simulate(n, times, self._start(start), self._rng(seed))   

    def sample_slices(self, times, n, rng, start):
        X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
        return np.transpose(X[:, 1:, :], (1, 0, 2))
    
    def sample(self, t, n=10000, start=None, seed=None):
        return self.sample_slices([t], n, self._rng(seed), self._start(start))[0]

There will be other things to add here, we will come to them in due time. Notice we kept `simulate` and `marginal` as empty promises for now. They will be defined properly in later subclasses. In fact, we will make another class now named `_IID` that will define `simulate` and `marginal` properly. 

If we are not defining them yet why do we even put it here inside `Process` to begin with? This is because there are other methods inside the class that are defined on top of it. For example, `sample` used `simulate` even though exact way to do the simulation is yet given (we will see them under subclass `_IID`). Similarly, there are other methods such as `pdf` and `center` (we did not define them yet but they are going to be a part of `Process`) that will need `marginal` even though `marginal` is going to be defined propery inside subclass `_IID`.

Also, keeping them declared here as empty promises are helpful since we can keep calling them back and define them in different ways in various subclasses. For example, in `_IID` the exact mechanism behind `simulate` is going to be different from the mechanism we see in `simulate` while making Brownian-motion (that is not $iid$ ofcourse). So, the point is, we will see several ways to define `marginal` and `simulate` across various sub-classes but operations we want to do with these simulated processes (like `sample`, `sample_slices`) and the marginals can still be done since they are defined in an unified manner under the same roof `Process`.

Say we want to create $iid$ Cauchy. Using the built process will give us all those properties to out cauchy process. So, doing it by hand using only `Process` looks like:

In [9]:
class cauchy(Process):
    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start                                    # time 0 = start point
        out[:, 1:, :] = rng.standard_cauchy((n, len(times)-1, 2))   # fresh noise afterwards
        return out

This works. Now if we want for Uniform we redo this:

In [10]:
class cauchy(Process):
    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start                                    # time 0 = start point
        out[:, 1:, :] = rng.uniform((n, len(times)-1, 2))   # fresh noise afterwards
        return out

Now we can do this for all processes we want such as exponential, Laplace, Student-t, lognormal, etc. This is a signal to make a shared engine for $iid$ processes. So we make another `class` named `_IID(Process)` and it has `Process` as argument since we do not want to miss out on the properties we built so far. This is also the reason why we call this the subclass.

In [11]:
class _IID(Process):
    """X_t independent across time; coordinates independent scipy distributions."""

    def __init__(self, frozen, seed=None, desc="iid"):
        super().__init__(seed)
        self.frozen = list(frozen)
        self._desc = desc

    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start
        for k in range(2):
            out[:, 1:, k] = self.frozen[k].rvs(size=(n, len(times) - 1), random_state=rng)
        return out

    def marginal(self, t, start, k):
        return self.frozen[k]

    def logpdf(self, X):
        return sum(self.frozen[k].logpdf(X[..., k]) for k in range(2))

As we said `simulate` will be defined again and again. Under `class _IID(Process)`, defining it is pretty easy; it is exactly same as the chunk of codes we were repeating again and again for Cauchy, uniform etc. 
1. `self.frozen[k].rvs(...)` replaces the hard-coded `rng.standard_cauchy`. It asks whatever distribution is stored for random draws, which is the whole point.
2. A loop over `k` in `range(2)`, because `self.frozen` is a list of two distributions, one for X (k=0) and one for Y (k=1). That lets X and Y have different parameters, for example X ~ Normal(0, 1) and Y ~ Normal(5, 2). Each fills its own slice `out[:, 1:, k]`. This gives us another degree of freedom hence.

As always `__init__` keeps in mind things about "self". For the base `class Process`, `__init__` was simply one ine long since it only needed `seed`. Here, for `class _IID(Process)` we will call back the seed from `Process` and add more details. `super().__init__(seed)` instaed of `self.seed = seed` because we are using the "parent's" `__init__` from `Process`. Lastly, `desc` is nothing mathematical, it is just a text label. If we type `print(p)`, Python shows the object’s `__repr__`, which returns `self._desc`. So with desc="cauchy", printing shows cauchy. This has no further deeper meaning.

Recall now, `marginal` was also declared as an empty promise inside `Process` like `simulate`. For $iid$ set up, distribution at any time $t$ is identical; so it is same as the stored distribtuion `self.frozen`. But keep track of the fact that it is not always going to be this easy. For example soon we will see the class for Brownian motion. There neither the `simulate` nor `marginal` will be this easy. In fact, this is the reason why we did the 1D processes first. Recall the marginal distribution of Brownian motion over time - the variance kept on increasing as we moved forward in time. So, here `marginal` at time $t$ will not be the stored distribtuion but a function of time $t$ itself. 

Lastly, the `logpdf` codes. This returns the log of the density at the points X. Because the X and Y coordinates are independent, the joint density is a product, $f(x, y) = f(x) · f(y)$. Taking logs turns the product into a sum, so we add the two log-densities. `X[..., k]` means “the k-th coordinate, whatever the other axes look like.”

Next we use `_IID` directly and see if it is still an inconvenience. For a Cauchy process we still have to write :

In [12]:
p = _IID([stats.cauchy(0, 1), stats.cauchy(0, 1)], seed=1, desc="cauchy")

`_IID` only ensures every $iid$ process now is one line long but we can make it even better. This was expected, I guess; the leading underscore I used in `_IID` is a hint that it is an internal engine, not something the user should touch. Now we give the user a cleaner front door to use named `iid`. Its job is to accept any scipy distribution and build the `_IID` for us. 

In [13]:
def iid(dist, seed=None):
    pair = list(dist) if isinstance(dist, (list, tuple)) else [dist, dist]
    return _IID(pair, seed, desc=f"iid({pair[0].dist.name})")

p = iid(stats.gamma(2), seed=1)
#notice how short it is now as compared to "p = _IID([stats.cauchy(0, 1), stats.cauchy(0, 1)], seed=1, desc="cauchy")"

`isinstance(input, (A, B))` is a standard code again. It asks if `input` is one of the two options `A` and `B`. If `input` is `A` or `B` it returns `True`. Above code says "If input is list or tuple, use them. Else repeat whatever we have twice."

What it still can’t do? `iid()` needs you to hand it a ready-made scipy distributions. For Cauchy we will write `iid(stats.cauchy(0, 1))`, which means knowing scipy’s parameter names.

In [14]:
p = iid([stats.norm(0, 1), stats.norm(5, 2)])    # a list, isinstace=True
p = iid((stats.norm(0, 1), stats.norm(5, 2)))    # a tuple works too, isinstance=True
p = iid(stats.gamma(1), seed=1)                  #not a tuple or a list, ininstance=False. So, it will be repeated twice

Now finally we have created all that we need for these $iid$ processes. We will generate $iid$ normal, $iid$ cauchy, $iid$ exponential, $iid$ uniform, $iid$ uniform, $iid$ student-t, $iid$ laplace and $iid$ lognormal. Every single one of these is going to be a sub-subclass since they will use `_IID` and that uses `Process`. 

We will start with making the process $iid$ Normal in 2D. In general they are defined using mean and variance where both are 2D vectors because the process is 2D. We want to add a feature that instead of `normal(mean=[0.0, 2.0], sigma=[1.0, 0.5])` if we simply write `normal(mean=0.0, sigma=1.0)` it will be able to recognize it as a 2D array by repeating the values as `normal(mean=[0.0, 0.0], sigma=[1.0, 1.0])`. That is where the first helper comes ito picture.

In [15]:
def _pair(v, name="parameter"):
    a = np.asarray(v, float)
    if a.ndim == 0:
        return np.repeat(a, 2)
    if a.shape == (2,):
        return a.copy()
    raise ValueError(f"{name} must be a scalar (used for both coordinates) or a length-2 sequence")

When entry is already 2-element list, `return a.copy()` is simply saying not to change anything about the entry and returning a copy of the entry (a copy instead of the original one to avoid re-writings). Everything else raises Error. 

In [16]:
class normal(_IID):
    """iid Normal(mean, sigma) noise at every time step (sigma = standard deviation)."""

    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")
        super().__init__([stats.norm(m[k], s[k]) for k in range(2)], seed,
                         f"normal(mean={fmt(m)}, sigma={fmt(s)})")


class cauchy(_IID):
    """iid Cauchy(loc, scale) noise (no mean, no variance, heavy tails)."""

    def __init__(self, loc=0.0, scale=1.0, seed=None):
        l, s = _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.cauchy(l[k], s[k]) for k in range(2)], seed,
                         f"cauchy(loc={fmt(l)}, scale={fmt(s)})")

class student_t(_IID):
    """iid Student-t(df, loc, scale) noise."""

    def __init__(self, df=3.0, loc=0.0, scale=1.0, seed=None):
        d, l, s = _pair(df, "df"), _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.t(d[k], l[k], s[k]) for k in range(2)], seed,
                         f"student_t(df={fmt(d)}, loc={fmt(l)}, scale={fmt(s)})")


class laplace(_IID):
    """iid Laplace(loc, scale) noise."""

    def __init__(self, loc=0.0, scale=1.0, seed=None):
        l, s = _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.laplace(l[k], s[k]) for k in range(2)], seed,
                         f"laplace(loc={fmt(l)}, scale={fmt(s)})")


There is a little change that we do with `exp`, `unif` and `lognorm`.

We will do a change of variable. The user will give a rate, but scipy’s exponential takes a scale, and `scale = 1/rate`. So the class will convert. This is done so that the idea converges with the notations we see in books, we generally write "exp(rate)" or $exp(\lambda)$ not $exp(\frac{1}{\lambda})$.

Similarly, `scipy` neeeds `start` and `width` for `uniform`. That is we are supposed to write "unif(a, b-a)". In books, we see this as "unif(a,b)" and we are going to use that for this library, so we are going to make an internal conversion once user uploads $(a, b)$.

A lognormal variable is `exp(Normal(mean, sigma))`. The user gives the `mean` and `sigma` of the underlying normal, but scipy wants the shape `sigma` and a `scale` equal to `exp(mean)`. So the class converts the mean with `np.exp`. Note that mean here is not the mean of the lognormal itself, only of the normal inside the exponential.

All three of these parameter changes are done to fall back to the more natural parameter set ups that we are used to seeing in books.

In [17]:
class exponential(_IID):
    """iid Exponential(rate) noise."""

    def __init__(self, rate=1.0, seed=None):
        r = _pair(rate, "rate")
        super().__init__([stats.expon(scale=1 / r[k]) for k in range(2)], seed,
                         f"exponential(rate={fmt(r)})")

class uniform(_IID):
    """iid Uniform(low, high) noise."""

    def __init__(self, low=0.0, high=1.0, seed=None):
        a, b = _pair(low, "low"), _pair(high, "high")
        super().__init__([stats.uniform(a[k], b[k] - a[k]) for k in range(2)], seed,
                         f"uniform(low={fmt(a)}, high={fmt(b)})")

class lognormal(_IID):
    """iid LogNormal noise: exp(Normal(mean, sigma))."""

    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")
        super().__init__([stats.lognorm(s[k], scale=np.exp(m[k])) for k in range(2)], seed,
                         f"lognormal(mean={fmt(m)}, sigma={fmt(s)})")

Most importantly keep track of the fact that we have now the inherited powers such as `simulate`, `marginal` and `logpdf` (from `_IID`), which further helps us inherit `sample`, `sample_slices`, `pdf`, `center` (last two are yet to be defined but they are built on top of `marginal`). 

The $iid$ family is almost done. The only loose end that remains are `pdf` and `center`. We said earlier that once `_IID` has marginal, the base class’s `pdf()` and `center()` work automatically, and we haven’t shown you those yet. The $iid$ is the easiest case to understand them in, so we’ll do `pdf` now and `center` right after that.

We already saw how `sample` and `sample_slices` looks at the law of the process $(X_t, Y_t)$ through a cloud of sample points. A cloud is only an approximation and that cannot be used to draw the density surface next to the paths as we wanted. If the law is known exactly, we can compute its density; then we will have a number saying how likely it is for the process to land near a given point $(x, y)$. High value means likely, low value means unlikely. The viewer uses this to draw the smooth surface next to the paths. For the iid processes, X and Y are independent. For independent things, the joint density is just the product of the two separate densities and we already know the separate densities. That was what `marginal` is for since it returns the distribution of one coordinate.

In [18]:
def pdf(self, t, x, y, start=None):
    if not self.independent:
        return None
    s = self._start(start)
    out = 1.0
    for k, c in enumerate((x, y)):
        d = self.marginal(t, s, k)
        f = d.pmf(np.rint(c)) if isinstance(d.dist, stats.rv_discrete) else d.pdf(c)
        out = out * f
    return out

Explaination:

1. `if not self.independent: return None`. The product rule only works for independent coordinates. `independent` is a flag defined at the top of the class, `True` by default. A process with correlated coordinates, like Brownian motion with `rho` ≠ $0$, switches it to `False`. Then this method gives up and returns `None`, meaning “no simple answer here”. Such processes can supply their own pdf instead. We will see them in the next subsections where we go to Jump processes and Brownian process families.
2. `for k, c in enumerate((x, y))` loops twice. The first time `k=0, c=x`, the second time `k=1, c=y`. So `k` is the coordinate number and `c` is the point’s value in that coordinate.
3. `d = self.marginal(t, s, k)` gets the distribution of coordinate `k`. For `_IID` this is just the stored frozen distribution.
4. The `f = ...` line picks the right function, and it’s the only tricky line. We have already seen what `isinstance` does. Some distributions are continuous (Normal, Cauchy), and for those `d.pdf(c)` gives a density. Others are discrete, meaning they only take whole-number values (for example the Poisson process later in the file), and for those the right quantity is `d.pmf(...)`, the probability of that exact value. `isinstance(d.dist, stats.rv_discrete)` asks “is this a discrete distribution?”. The `np.rint(c)` rounds `c` to the nearest whole number first, so a point like $2.0000001$ doesn’t get a probability of zero.

5. Since it will loop twice and we start at `out = 1.0` we get the following: $$1.0 \longrightarrow 1.0* f(x) \longrightarrow \{1.0*f(x)\}*f(y)$$

Why this lives in the base class?

It is written once and uses only `marginal`. Any process that can say “here is the distribution of coordinate k at time t” gets pdf for free, as long as its coordinates are independent. For iid, that’s all the classes at once.

Lastly, `center` is where the cloud is centered. Basically the mean and we will not do the main hard work since that will be handled by `scipy` itself. We will just ask for the mean.

In [19]:
def center(self, t, start=None):
    s = self._start(start)
    if t <= 0:
        return s.copy()
    out = []
    for k in range(self.dim):
        d = self.marginal(t, s, k)
        m = d.mean()
        out.append(m if np.isfinite(m) else d.median())
    return np.array(out, float)

Explaination:
1. `s = self._start(start)` cleans the starting point, as always.
2. `if t <= 0: return s.copy()` handles time zero (or earlier). At time 0 nothing has happened yet, so the process is sitting exactly at its start point. `.copy()` hands back a fresh array so nobody can accidentally change `s` as we often do to avoid re-writings.
3. `for k in range(self.dim)` loops over the two coordinates, k = 0 for X and k = 1 for Y as always.
4. `d = self.marginal(t, s, k)` gets the distribution of that coordinate. It’s the same marginal we’ve now used three times.
5. As mentioned earlier, `scipy` will do the main job. `m = d.mean()` asks `scipy` for the average of that distribution.
6. `m if np.isfinite(m) else d.median()` is the interesting line. It means “use the mean, unless it isn’t a real finite number, in which case use the median.” This is specifically done keeping Cauchy in mind. This `center` is NOT just for calculating mean for the viewer, it also tells us where to aim the camera later on. So `nan` mean values returning the value "infinity" is not good enough, we need a point to focus.
7. `out.append()` simply appends the two center_x and center_y into `[center_x, center_y]`. That is why here we started with empty `out = []` whereas for `pdf` we had `out = 1.0` because there we were taking products in loop.

The full updated code as of now is the following:

In [20]:
from __future__ import annotations

import numpy as np
from scipy import stats

# ----------------------------------------------------------------------------------------- helpers
def _pair(v, name="parameter"):
    a = np.asarray(v, float)
    if a.ndim == 0:
        return np.repeat(a, 2)
    if a.shape == (2,):
        return a.copy()
    raise ValueError(f"{name} must be a scalar (used for both coordinates) or a length-2 sequence")

# --------------------------------------------------------------------------------------- base class
class Process:
    """Base class: every process knows how to simulate itself and (where possible) its law."""

    name = "process"
    dim = 2
    default_start = (0.0, 0.0)
    independent = True
    _desc = "process"

    def __init__(self, seed=None):
        self.seed = seed

    def __repr__(self):
        return self._desc

    def _start(self, start):
        """Validates or assigns default starting vector (0.0, 0.0)."""
        s = self.default_start if start is None else start
        s = np.asarray(s, float)
        if s.ndim == 0:
            s = np.repeat(s, self.dim)
        if s.shape != (self.dim,):
            raise ValueError(f"start must have {self.dim} coordinates")
        return s

    def _rng(self, seed=None):
        return np.random.default_rng(self.seed if seed is None else seed)

    #------------------------------------------------------------------ to be provided by subclasses
    def simulate(self, n, times, start, rng):
        """Paths of shape (n, len(times), dim); times[0] must be 0 and X_0 = start."""
        raise NotImplementedError

    def marginal(self, t, start, k):
        """Frozen scipy distribution of coordinate k at time t (when known in closed form)."""
        raise NotImplementedError

    #-----------------------------------------------------------------------------------------------

    def pdf(self, t, x, y, start=None):
        """Closed-form density (pmf for lattice laws) of (X_t, Y_t) at (x, y), or None.
        
        Product rule for independent coordinates: f(x, y) = f(x) * f(y)
        """
        if not self.independent:
            return None
        s = self._start(start)
        out = 1.0
        for k, c in enumerate((x, y)):
            d = self.marginal(t, s, k)
            f = d.pmf(np.rint(c)) if isinstance(d.dist, stats.rv_discrete) else d.pdf(c)
            out = out * f
        return out

    def center(self, t, start=None):
        """Mean of X_t (median if the mean does not exist, e.g. Cauchy)."""
        s = self._start(start)
        if t <= 0:
            return s.copy()
        out = []
        for k in range(self.dim):
            d = self.marginal(t, s, k)
            m = d.mean()
            out.append(m if np.isfinite(m) else d.median())
        return np.array(out, float)
    
    def sample_slices(self, times, n, rng, start, fine=None):
        """Simulates up to slice times and extracts the state at those times. Exact for all built-in processes."""
        X = self.simulate(n, np.r_[0.0, np.asarray(times, float)], start, rng)
        return np.transpose(X[:, 1:, :], (1, 0, 2))
        # Transpose from (n, slices, dim) -> (slices, n, dim)

    def paths(self, n=20, T=1.0, steps=200, start=None, seed=None):
        """Simulate n paths on [0, T]; returns (times, X) with X of shape (n, steps+1, 2)."""
        times = np.linspace(0.0, T, steps + 1)
        return times, self.simulate(n, times, self._start(start), self._rng(seed))

    def sample(self, t, n=10000, start=None, seed=None):
        """n independent samples of X_t, shape (n, 2) ie, at fixed time."""
        return self.sample_slices([t], n, self._rng(seed), self._start(start))[0]

# ------------------------------------------------------------------------------------ iid processes
class _IID(Process):
    """X_t independent across time; coordinates independent scipy distributions."""

    def __init__(self, frozen, seed=None, desc="iid"):
        super().__init__(seed)
        self.frozen = list(frozen)
        self._desc = desc

    def simulate(self, n, times, start, rng):
        out = np.empty((n, len(times), 2))
        out[:, 0, :] = start
        for k in range(2):
            out[:, 1:, k] = self.frozen[k].rvs(size=(n, len(times) - 1), random_state=rng)
        return out

    def marginal(self, t, start, k):
        return self.frozen[k]

    def logpdf(self, X):
        return sum(self.frozen[k].logpdf(X[..., k]) for k in range(2))


def iid(dist, seed=None):
    """iid(scipy.stats.gamma(2)) or iid([frozen_x, frozen_y]): any scipy distribution."""
    pair = list(dist) if isinstance(dist, (list, tuple)) else [dist, dist]
    return _IID(pair, seed, desc=f"iid({pair[0].dist.name})")


class normal(_IID):
    """iid Normal(mean, sigma) noise at every time step (sigma = standard deviation)."""

    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")
        super().__init__([stats.norm(m[k], s[k]) for k in range(2)], seed,
                         f"normal(mean={fmt(m)}, sigma={fmt(s)})")


class cauchy(_IID):
    """iid Cauchy(loc, scale) noise (no mean, no variance, heavy tails)."""

    def __init__(self, loc=0.0, scale=1.0, seed=None):
        l, s = _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.cauchy(l[k], s[k]) for k in range(2)], seed,
                         f"cauchy(loc={fmt(l)}, scale={fmt(s)})")


class exponential(_IID):
    """iid Exponential(rate) noise."""

    def __init__(self, rate=1.0, seed=None):
        r = _pair(rate, "rate")
        super().__init__([stats.expon(scale=1 / r[k]) for k in range(2)], seed,
                         f"exponential(rate={fmt(r)})")


class uniform(_IID):
    """iid Uniform(low, high) noise."""

    def __init__(self, low=0.0, high=1.0, seed=None):
        a, b = _pair(low, "low"), _pair(high, "high")
        super().__init__([stats.uniform(a[k], b[k] - a[k]) for k in range(2)], seed,
                         f"uniform(low={fmt(a)}, high={fmt(b)})")


class student_t(_IID):
    """iid Student-t(df, loc, scale) noise."""

    def __init__(self, df=3.0, loc=0.0, scale=1.0, seed=None):
        d, l, s = _pair(df, "df"), _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.t(d[k], l[k], s[k]) for k in range(2)], seed,
                         f"student_t(df={fmt(d)}, loc={fmt(l)}, scale={fmt(s)})")


class laplace(_IID):
    """iid Laplace(loc, scale) noise."""

    def __init__(self, loc=0.0, scale=1.0, seed=None):
        l, s = _pair(loc, "loc"), _pair(scale, "scale")
        super().__init__([stats.laplace(l[k], s[k]) for k in range(2)], seed,
                         f"laplace(loc={fmt(l)}, scale={fmt(s)})")


class lognormal(_IID):
    """iid LogNormal noise: exp(Normal(mean, sigma))."""

    def __init__(self, mean=0.0, sigma=1.0, seed=None):
        m, s = _pair(mean, "mean"), _pair(sigma, "sigma")
        super().__init__([stats.lognorm(s[k], scale=np.exp(m[k])) for k in range(2)], seed,
                         f"lognormal(mean={fmt(m)}, sigma={fmt(s)})")

## 2.1.2 Brownian family

For Brownian motions, $iid$ is not good enough. In $iid$, every time point is a fresh draw and the process has no memory. Brownian motion is the opposite since it remembers where it was. Each step starts from the current position and moves a little. So we will create a new class `bm` where it will have its own `simulate` and `marginal`. Making 1D Brownian motion was already explored in *Chapter 01*. Here, for 2D Brownian motion we simply need two 1D Brownian motions where they can independent (standard choice) or they might have some correlation (so that this library has more interesting processes to plot). To create the 2D Brownian motion with correlation we will need a helper `_vol` (so far our only helper was `_pair`).

So far we’d draw X’s noise and Y’s noise separately, so they’d be unrelated. But in the real world (and in the stock part of this file that we are yet to see) two things often move together. When X gets a big random push up, Y tends to get one too. The strength of that tendency is the correlation `rho`:
1. rho = 0: unrelated
2. rho = 1: always move together
3. rho = -1: always move opposite
   
We need noise for X and Y whose spreads are `sigma_x` and `sigma_y` and whose correlation is `rho`. The trick is to simply mix independent noises now since we can only generate independent standard normals, `Z1` and `Z2`; so the only way to build correlated noise is by mixing them. $$noise_x = \sigma_x · Z1
$$
$$noise_y = \sigma_y · ( \rho · Z1  +  \sqrt{1 − \rho²} · Z2 )$$

Y’s noise borrows a share rho of X’s noise `Z1`, and the rest comes from fresh noise `Z2`. The above formulae ensures correlation but keeps the variance still $\sigma_x$ and $\sigma_y$. `A` holds all of the correlation and spread information in one object.
$$L = \begin{bmatrix} 
1 & 0 \\ 
\rho & \sqrt{1 - \rho^2} 
\end{bmatrix} \qquad
A = \begin{bmatrix} 
\sigma_x \cdot 1 & 0 \\ 
\sigma_y \cdot \rho & \sigma_y \cdot \sqrt{1 - \rho^2} 
\end{bmatrix}$$
This is what the helper `_vol` will do.

In [21]:
def _vol(sigma, rho):
    if not -1 < rho < 1:
        raise ValueError("rho must lie strictly between -1 and 1")
    L = np.array([[1.0, 0.0], [rho, np.sqrt(1 - rho ** 2)]])
    return sigma[:, None] * L  # A = diag(sigma) L, covariance Sigma = A A^T

We know `[:, None]` means "for each fixed row one by one, columns are not running". We know the recipe for a single coordinate. Now we need it for *n paths, 2 coordinates, with the correlation matrix `A`* and this is where the next helper `_gauss_paths` come into picture.

In [22]:
def _gauss_paths(n, times, drift, A, rng):
    dt = np.diff(times)
    z = rng.standard_normal((n, len(dt), 2))
    inc = np.sqrt(dt)[None, :, None] * (z @ A.T) + drift[None, None, :] * dt[None, :, None]
    return np.concatenate([np.zeros((n, 1, 2)), np.cumsum(inc, axis=1)], axis=1)

Let us discuss this in details since this involves 3D arrays.

Explanation:
1. If `times` has $T$ points (say, $[0.0, 0.25, 0.5, 1.0]$), `np.diff` subtracts each element from the next one, producing an array of length $T - 1$: $[0.25, 0.25, 0.5]$. This is because Brownian motion evolves in discrete intervals. We need the exact duration of each step because both the drift and the random volatility scale with time. If `times` has N points, dt has N−1 numbers.
2. `z = rng.standard_normal((n, len(dt), 2))` helps us draw the required noises for `n` many paths, `len(dt)` many time steps and then do this twice.
3. `inc = np.sqrt(dt)[None, :, None] * (z @ A.T) + drift[None, None, :] * dt[None, :, None]` is the core math. This is basically doing $$\Delta X_t = \mu \Delta t + \sqrt{\Delta t} \, A Z_t$$
   $\bullet$ Because `dt` is a 1D array of shape `(T-1,)` and `drift` is a 1D array of shape `(2,)`, NumPy cannot directly multiply them with a 3D array of shape `(n, T-1, 2)` that `z` has. Inserting `None` adds dummy axes of size 1 so their shapes align. We will come to this, but let us first see what shape will be needed. So, let us understand `z@A.T` first. Do the calculation $$A\begin{bmatrix} z1 \\ z2 \end{bmatrix} = \begin{bmatrix} \sigma_x & 0 \\ \sigma_y \rho & \sigma_y \sqrt{1 - \rho^2} \end{bmatrix}\begin{bmatrix} z1 \\ z2 \end{bmatrix}=\big(\begin{bmatrix} z1 & z2 \end{bmatrix} A^{T}\big)^T$$
   $\bullet$ We wrote `noise = A · z` in the mathematical formulae. That treats `z` as a column of two numbers. Here, in codes `z` stores each pair as a row (the last axis of size 2), and for rows the same mixing is written `z @ A.T`. `z` stores $T-1$ many $[z1 \quad z2]$ from top to bottom is the natural order of shape (T-1, 2), right?
   $$
\begin{aligned}
T - 1 = 0 \quad : \quad &\begin{bmatrix} Z_1^0 & Z_2^0 \end{bmatrix} \\[4pt]
T - 1 = 1 \quad : \quad &\begin{bmatrix} Z_1^1 & Z_2^1 \end{bmatrix} \\[4pt]
T - 1 = 2 \quad : \quad &\begin{bmatrix} Z_1^2 & Z_2^2 \end{bmatrix} \\[4pt]
T - 1 = 3 \quad : \quad &\begin{bmatrix} Z_1^3 & Z_2^3 \end{bmatrix} \\[4pt]
T - 1 = 4 \quad : \quad &\begin{bmatrix} Z_1^4 & Z_2^4 \end{bmatrix} \\[4pt]
T - 1 = 5 \quad : \quad &\begin{bmatrix} Z_1^5 & Z_2^5 \end{bmatrix}
\end{aligned}
$$
Then `n` many copies of this will give other paths. More importantly because of this structure we have to use `z@A.T` instead of `A@z`. We know `NumPy`’s `@` operator (np.matmul) follows a strict, deterministic rule when given arrays with more than 2 dimensions; it always treats the last two axes as 2D matrices, and treats all leading axes as a "batch" of independent matrices. So, `z@A.T` does the desired calculations:
$$\begin{bmatrix} Z_1^0 & Z_2^0 \\
                  Z_1^1 & Z_2^1 \\
                  \vdots & \vdots\\
\end{bmatrix}\begin{bmatrix} \sigma_x & \sigma_y \rho \\ 0 & \sigma_y \sqrt{1 - \rho^2} \end{bmatrix}
$$
This has shape (T-1, 2)x(2, 2)=(T-1, 2). This will happen `n` many times and hence we will have the shape (n, T-1, 2). So, when we multiply $\sqrt{\Delta t}$ to $AZ_t$, in codes, it will try to multiply `T-1` array to (n, T-1, 2). Unlike `@`, `*` performs element-wise multiplication. When the shapes don't match, NumPy applies its broadcasting rules from right to left: two dimensions are compatible if they are equal, or if one of them is 1. So, we can make the shape of `np.sqrt(dt)` from `T-1` to (1, T-1, 1) by using `np.sqrt(dt)[None, :, None]`. What we have drawn below is only the `np.sqrt(dt)[:, None]` part. The $0^{th}$ axis is not important now. Basically, the following thing will happen `n` times.

$$\begin{bmatrix} dt1 \\
dt2\\
\vdots\\
\end{bmatrix}*\begin{bmatrix} Z_1^0 & Z_2^0 \\
                  Z_1^1 & Z_2^1 \\
                  \vdots & \vdots\\
\end{bmatrix}@\begin{bmatrix} \sigma_x & \sigma_y \rho \\ 0 & \sigma_y \sqrt{1 - \rho^2} \end{bmatrix} \qquad\text{CANNOT PROCEED with *}
$$
$$\begin{bmatrix} dt1 & dt1\\
dt2 & dt2\\
\vdots & \vdots\\
\end{bmatrix}*\begin{bmatrix} Z_1^0 & Z_2^0 \\
                  Z_1^1 & Z_2^1 \\
                  \vdots & \vdots\\
\end{bmatrix}@\begin{bmatrix} \sigma_x & \sigma_y \rho \\ 0 & \sigma_y \sqrt{1 - \rho^2} \end{bmatrix} =
\begin{bmatrix} (dt1) (Z_1^0) & (dt1)(Z_2^0) \\
                (dt2) (Z_1^1) & (dt2)(Z_2^1) \\
                  \vdots & \vdots\\
\end{bmatrix}@\begin{bmatrix} \sigma_x & \sigma_y \rho \\ 0 & \sigma_y \sqrt{1 - \rho^2} \end{bmatrix}
$$
What we wrote above has shape (T-1, 2) but then when we do it `n` times we will have the shape (n, T-1, 2). If we want to add $ \mu \Delta t$ now, we have to make it the same shape since `+` will act element wise. `dt` with shape `(T-1, )` will become `dt[None, :, None]` that has shape `(1, T-1, 1)` and `drift` that had shape `(2, )` will become `drift[None, None, :]` that will have shape `(1, 1, 2)`. Where to put `None` and `:` is decided based on what physical quantity they are. Times is supposed to be the 1st entry, to we have it in 1st position where as `drift` is basically $(\mu_x, \mu_y)$, so it should be in the 2nd entry. None of themm will be in $0^{th}$ position. Now `*` will do the required operation; `drift[None, None, :] * dt[None, :, None]` makes `(1, 1, 2)+(1, T-1, 1)`= `(1, T-1, 2)`. Then `+` will make `(n, T-1, 2)+(1, T-1, 2)` to become `(n, T-1, 2)` in `np.sqrt(dt)[None, :, None] * (z @ A.T) + drift[None, None, :] * dt[None, :, None]`. Now, with this shape (n, T-1, 2) we simply need to do `cumsum` along `axis=1` to get the n-many paths.

This concludes the helper `_gauss_paths` using another helper `_vol` that gave us $A$.

Let us now move on the actual class for Brownian motion named `bm`. We now have both ingredients, `_vol` for the mixing matrix and `_gauss_paths` for the simulation. The `bm` class just assembles them. It’s a `Process`, like `_IID`, so it must supply `simulate` and `marginal`.

In [23]:
class bm(Process):
    
    def __init__(self, mu=0.0, sigma=1.0, seed=None, rho=0.0):
        super().__init__(seed)
        self.mu, self.sigma, self.rho = _pair(mu, "mu"), _pair(sigma, "sigma"), float(rho)
        self.A = _vol(self.sigma, self.rho)
        self.Sigma = self.A @ self.A.T
        self.independent = self.rho == 0.0
        self._desc = ...

    def _gauss_params(self):
        return self.mu, self.A

    def simulate(self, n, times, start, rng):
        return start + _gauss_paths(n, times, self.mu, self.A, rng)

    def marginal(self, t, start, k):
        return stats.norm(start[k] + self.mu[k] * t, np.sqrt(self.Sigma[k, k] * t))

This is same as `_IID` and it stores what the object will need later. `super()` and `_pair` are exactly as before and they uses parent-`seed` & turn it into a $2$ tuple if not provided already, respectively. `self.A` will make the mixing matrix with provided `sigma` and `rho`.

As for `simulate`, all the hard work is done by the helper. `_gauss_paths` returns paths starting at zero, and adding `start` shifts them to begin at the right place. So `bm.simulate` is one line. The `drift self.mu` and the mixing `matrix self.A` are exactly what `_gauss_paths` in `__init__` asks for.

As for the `marginal`, it will answer “what is the distribution of coordinate k at time t?”, the same question `_IID.marginal` answered, but now the answer depends on `t`. A sum of normal steps is itself normal, so each coordinate at time `t` is a Normal with
1. mean `start[k] + mu[k] · t`: begin at the start and drift for time t
2. standard deviation `sqrt(Sigma[k, k] · t)`: the variance of Brownian motion grows linearly in `t`, and `Sigma[k, k]` is the variance per unit time for that coordinate (since recall that is what the diagonals of a covariance matrix gives). Taking the square root gives the standard deviation, which is what scipy wants.

We also had `pdf` in `Process` but the base class’s `pdf` isn’t enough anymore when the coordinates X and Y can be dependent. So bm needs a density that treats `(x, y)` together. That is the bivariate normal density, and it’s what `_bvn_pdf` computes. The general rule, in matrix form, is $$q = dᵀ · Σ⁻¹ · d$$ where `d = (dx, dy)` and $Σ$ is the covariance matrix. This is a direct generalization of 1D Gaussian. In 1D, $Σ = σ²$, and $d · (1/σ²) · d = d²/σ²$, the same `q`. So $Σ⁻¹$ is just “1/variance” for matrices. Expand the matrix multiplication to get the formulae used in the following codes.

In [24]:
def _bvn_pdf(x, y, mean, cov):
    dx, dy = x - mean[0], y - mean[1]
    det = cov[0, 0] * cov[1, 1] - cov[0, 1] ** 2
    q = (cov[1, 1] * dx * dx - 2 * cov[0, 1] * dx * dy + cov[0, 0] * dy * dy) / det
    return np.exp(-0.5 * q) / (2 * np.pi * np.sqrt(det))

def pdf(self, t, x, y, start=None):
    s = self._start(start)
    return _bvn_pdf(x, y, s + self.mu * t, self.Sigma * t)

This overrides the base class’s `pdf`. When you call `bm(...).pdf(...)`, Python uses this version and not the one from `Process`. It supplies the two inputs `_bvn_pdf` needs namely `s + self.mu * t` and `self.Sigma * t`. Even if the coordinate processes are independent, we do not need to fall back to base class's `pdf` since the new method uses the actual form which will automatically make the off diagonal entries in $\Sigma$ to be $0$.

The next three methods are `cumulant`, `esscher` and `theta_star`. 

The tilt: To change the law, we reweight it. Take the original density $f(x)$ and multiply it by $e^{\theta x}$. Then $$\text{tilted density}  ∝  e^{\theta x} · f(x)$$
So, $\theta$ is a knob that slides the law left or right. This reweighting is the Esscher transform. After multiplying, the density no longer adds up to 1, so we divide by a constant to fix that. That constant is $E[e^{\theta X}]$, called the *moment generating function (MGF)*. The key fact for this new part is that Brownian motion can be tilted using $\theta$ and hence `esscher` can return a new `bm` object. Two requirements for this application will be:
1. `has_mgf`: the MGF $E[e^{\theta X}]$ must actually exist. For Cauchy it doesn’t (its tails are too heavy, so the expectation is infinite). Cauchy can’t be tilted, and the file prints `NO_MGF_MSG` for it.
2. `is_levy`: the process must have independent, stationary increments (a Lévy process). Only then does tilting at time t stay consistent across all times and keep the same family. Brownian motion qualifies, but `bm_bridge` doesn’t, which is why `NOT_LEVY_MSG` exists.

Next there will be `esscher_space` that says where the tilt acts. We will add this in the `Process` class's attribute with default `None` that will mean that *Esscher's tilt* isn't supported (for example in `_IID`, `bm_bridge`, `cauchy_process`). We will have two other forms. `esscher_space="raw"` means on X itself (like `bm`, `poisson`, `gamma_process`), and `"log"` means tilt acts on `log (X_t / start)` instead of on `X` directly (for stocks, whose logarithm is the Lévy part but we haven't yet made the stock's class).

Why two different spaces `"raw"` and `"log"` is needed? Tilting by `e^{θ·(state)}` only works nicely when the “state” is a Lévy process, meaning independent, stationary increments (Part 17). Then the tilted process stays in the same family, and the normalizing constant is `e^{t·κ(θ)}` (we will talk in details about `κ` soon). For `bm`, `poisson`, `gamma_process` they are Lévy themselves whereas for stocks `S` is not Lévy but `log S` is.

REMARK: The only technicality is with `cauchy_process`. It gives `is_levy` but has to `MGF`, so tilt can not act.

What the code will need? To tilt a process, the code needs to answer two things:
1. How does the normalizing constant $E[e^{\theta X_t}]$ depend on $\theta$ and t? That’s the `cumulant` method.
2. What is the new process after tilting? That’s the `esscher` method.
3. Which $\theta$ makes the `drift` exactly $0$? That’s `theta_star`.

When we tilt by $e^{\theta x}$, we must divide by $E[e^{\theta X}]$ so the total probability stays 1. For Brownian motion this is easy to compute, because the increment over time t is Normal, and a Normal has a known MGF. 

In 1D, for $X_t \sim \text{Normal}(\mu t, {\sigma}^2t)$: $$E[e^{\theta X_t}] = exp( t · (\theta \mu + {(1/2)}{\theta}^2{\sigma}^2) )$$
The time t factors out, so we write the bracket once as the "cumulant" $$κ(\theta) = \theta μ + {(1/2)}{\theta}^2{\sigma}^2$$
In general $E[e^{\theta X_t}] = e^{t·κ(\theta)}$, which means “cumulant” is just the log of the normalizing constant, per unit time.

In 2D, $\theta$ is a pair $(\theta_x, \theta_y)$, and the formula becomes the line in the code:
1. `self.mu @ theta` is `μx·\theta x + μy·\theta y`, the drift part.
2. `0.5 * theta @ self.Sigma @ theta` is `½ θᵀ Σ θ`, the 2-D version of `½θ²σ². Σ` appears here because it holds the variances and the correlation.

It’s wrapped in `float(...)` just so it returns a plain number. You’ll see it used later in `weights`, in the expression `t * self.cumulant(th)`, exactly the `t · κ(θ)` from above.

In [25]:
def cumulant(self, theta):
    return float(self.mu @ theta + 0.5 * theta @ self.Sigma @ theta)

def esscher(self, theta):
    return bm(self.mu + self.Sigma @ np.asarray(theta, float), self.sigma, self.seed, self.rho)

def theta_star(self):
    s1 = self.Sigma @ np.ones(2)
    return float(-(s1 @ self.mu) / (s1 @ s1))

We will later on add a Plotly plot that turns this `θ` into a slider, so the entire process of removing drift will become an interactive live page where we can see the drift being removed in real time. Then `esscher` gives a brand new `bm` object.

*REMARK*: When we get to plotting, we will use a single slider for `θ` meaning the true object that we will be fine-tuning is $(\theta, \theta)$ instead of  $(\theta_x, \theta_y)$. This is just a design choice for ease of building. The math underneath is already vector-valued. `esscher(theta)` takes a length-2 `θ`, `cumulant(θ)` will work on vectors too. So, moving our slider (once we get there) to some value of `θ` will add the same tilt to both coordinates.

It will have drawbacks ofcourse. When the two coordinates have different $\mu$ or $\sigma$, or $\rho \neq 0$, that vector is not along $(1, 1)$. My `“snap to θ*”` button then only gives a least-squares compromise, and one coordinate keeps some drift. It is correct in the symmetric case and only approximate otherwise, which is a real limitation, not just a labelling issue. However, since this library is meant for visual aids let us stick with this for now. We will have the improved version some day later.

Lastly, `theta_star` will be the most optimal drift-cancellation. We want the tilt that makes the drift as close to zero as possible. Since `mu_new = mu + Σθ`, we’d like $\Sigma \theta = −\mu$. By usual optimization argument we will find the critical point and get hold of the solution. Setting the derivative wrt $\theta$ of $|\mu + \theta s_1|^2$ to $0$, where $s_1 = \Sigma \begin{bmatrix}1 \\ 1\end{bmatrix}$, we get $\theta* = -(s_1 \cdot \mu) / (s_1 \cdot s_1)$. That is exactly what th code reads.

Next we will make the class for Brownian bridge. A Brownian motion step could be drawn independently of the past. A bridge can’t, because each step must know where we are now and how much time is left. So this `simulate` uses a loop and has no `cumsum`.The one rule is, given the current position `cur` at time `t0`, the next position at `t1` is Normal with:
```text
rem = T − t0               (time remaining)
dt  = t1 − t0              (this step's length)

mean = cur + (end − cur) · dt / rem
var  = sigma² · dt · (T − t1) / rem
```

In [26]:
def simulate(self, n, times, start, rng):
    if times[-1] > self.T + 1e-12:
        raise ValueError(f"a bridge only lives on [0, {self.T:g}]")
    X = np.empty((n, len(times), 2))
    X[:, 0, :] = cur = np.broadcast_to(start, (n, 2)).copy()
    for i in range(1, len(times)):
        t0, t1 = times[i - 1], times[i]
        rem, dt = self.T - t0, t1 - t0
        if rem <= 1e-12:
            cur = np.broadcast_to(self.end, (n, 2)).copy()
        else:
            mean = cur + (self.end - cur) * dt / rem
            var = self.sigma ** 2 * dt * max(self.T - t1, 0.0) / rem
            cur = mean + np.sqrt(var) * rng.standard_normal((n, 2))
        X[:, i, :] = cur
    return X

Explainers:
1. Allocate X, the same (n, times, 2) shape as always.
2. `cur = np.broadcast_to(start, (n, 2)).copy()` makes `n` copies of the start point, one per path, so that `cur` has shape `(n, 2)` and holds every path’s current position. The chained `X[:, 0, :] = cur = ...` does two things at once; it stores that in the time-0 column and also names it `cur`.
3. The loop walks through the time points. For each step it computes `rem` and `dt` and applies the rule above.
4. The `rem <= 1e-12` branch covers the case where we are already at time `T`, with nothing remaining (dividing by rem would blow up). The path just sits at `end`.
5. `mean + np.sqrt(var) * rng.standard_normal((n, 2))` draws the new position: the mean plus noise scaled to the right spread. `sigma ** 2` is an array of two values, so each coordinate gets its own spread.
6. `max(self.T - t1, 0.0)` protects against `T - t1` being a hair negative from rounding, which would make the variance negative and the square root fail!!
7. `X[:, i, :] = cur` records the positions at time `i`.

Now it is time for `bm_bridge.marginal`. The question is again “what is the distribution of coordinate k at time t?” Here the answer is Normal again, with a different mean and variance than `bm` though since we will have to collapse back. The mean is a staright line and variance is an arch. the variance will be $0$ at start and at end, it will be maximum at $t=T/2$.

In [27]:
def marginal(self, t, start, k):
    mean = start[k] + (t / self.T) * (self.end[k] - start[k])
    var = self.sigma[k] ** 2 * t * (self.T - t) / self.T
    return stats.norm(mean, max(np.sqrt(max(var, 0.0)), 1e-3 * self.sigma[k]))

What we get for free here?


There is no pdf override needed here. The bridge has no `rho`, so the coordinates are independent, `independent = True` stays from the base class, and the base class’s product rule `f(x)*f(y)` works automatically from this `marginal`.

This concludes the discussion on Brownian family. We move on to the next bunch of processes.

## 2.1.3 The Jump Processes

Under Jump processes we will have `poisson`, `gamma_process` and `cauchy_process`.

Let us start with `poisson`. The same pattern will be followed. We will make `simulate` and `marginal`.

A Poisson process counts events: customers arriving, phone calls, radioactive emmisions. The path starts at some number and goes up by 1 each time an event happens. It never goes down, and it only takes whole-number values, so the path looks like a staircase. The one parameter is the rate $\lambda$ and it is the average number of events per unit time. With rate=3, you expect about 3 events per time unit.

Compared with what we’ve seen:
1. `_IID`: fresh noise every time, no memory.
2. `bm`: steps added up, with Normal steps.
3. `poisson`: steps added up, with Poisson steps.

So `poisson` will have the same structure as `bm`, and only the step distribution should be different. The step distribution is defined as over a time gap `dt`, the number of new events is a Poisson random number with mean `rate*dt`. A longer gap means more events on average, and the counts over different gaps are independent. That is the Poisson counterpart of Brownian motion’s rule “variance proportional to dt”.

In [28]:
def simulate(self, n, times, start, rng):
    dt = np.diff(times)
    inc = rng.poisson(self.rate[None, None, :] * dt[None, :, None], size=(n, len(dt), 2))
    return start + np.concatenate([np.zeros((n, 1, 2)), np.cumsum(inc, axis=1)], axis=1)

We have already seen all the ingrediants. 

Notice there is no `A` matrix and no correlation here. The two coordinates count events independently, so the base class’s `independent = True` stays. That makes alot of stuffs easily available again.

In [29]:
def marginal(self, t, start, k):
    return stats.poisson(self.rate[k] * t, loc=start[k])

The total number of events by time `t` is a sum of independent Poisson counts. A sum of Poissons is again Poisson, with the means added: `rate · t`. The `loc=start[k]` shifts the whole distribution by the starting value, so the count begins from start.

This is where the discrete case from earlier shows up. The base class’s `pdf` checks `isinstance(d.dist, stats.rv_discrete)` and uses `pmf` with `np.rint` for exactly this kind of distribution. `poisson` is the first process in this file that triggers that branch. `center` also works; the mean is `start + rate · t`.

The levels on the class are:
```text
is_levy = True
has_mgf = True
esscher_space = "raw"
```
Increments are independent and stationary, so it’s a Lévy process. Its MGF exists, so it can be tilted, and the tilt acts on the raw values. We’ll see the actual methods next.

Now `poisson`'s `cumulant` and `esscher` will be similar to before (infact the same goal but the algebra will change since the processes are different). `cumulant` is the normalizing constant:
$$\kappa(\theta) = \lambda_x (e^{\theta_x} - 1) + \lambda_y (e^{\theta_y} - 1)$$
because in 2D, we have this formulae $e^{\theta_x X + \theta_y Y}$ inside Expectation. `esscher` works similarly. 

Note that `poisson` will have no `theta_star`, so the base class’s version returns `None`. A Poisson count only goes up, and multiplying the rate by $e^\theta$ can shrink it but never make it exactly zero, so there’s no tilt that “removes the drift” the way there was for Brownian motion.

Next is `gamma_process`. A gamma process is another “steps added up” process, like `bm` and `poisson`, but each step is Gamma-distributed. Like Poisson, it only goes up, since a gamma variable is never negative. Unlike Poisson, it grows continuously, in tiny positive increments, so the path is a smooth-ish rising curve and not a staircase.

It has two parameters per coordinate:
1. `a`: the shape rate
2. `b`: the rate (inverse scale)
The defining rule is that over a gap `dt`, the step is `Gamma(shape = a·dt, rate = b)`. As with `bm` and `poisson`, longer gaps produce bigger steps, and the shape parameter `a·dt` is what carries the dependence on `dt`.

The only wrinkle here is we have to ensure $b-\theta$ is positive. So, we will have a `theta_domain` here. The rest is again based on similar ideas and we skip it here.

## 2.1.4 Stocks

Why not model a stock price as Brownian motion? Two reasons:
1. A price can’t go negative. Brownian motion can.
2. Moves should be proportional to the price. A stock at 1000 moves by much larger amounts than a stock at 1. Brownian motion adds noise of the same size wherever it is.

So instead of modeling the change, we model the percentage change:
```text
change in price = price × (small drift + small noise)
```
We will have $$d\tilde{S}_i = ( (\mu_i − r) dt + \Sigma_j A_{ij} dB_j ) \tilde{S}_i$$
where $\tilde{S}$ is the discounted price, with stock growth rate $\mu$ and drift $\mu - r$, $dt$ and $dB$ are tiny time step and tiny Brownian increment respectively, $A$ is mixing matrix, same as before, so the two stocks’ noises can be correlated through $\rho$. Lastly, we multiply by current price $\tilde{S}_i$. The solution to this SDE is 
$$\tilde{S}_t = \tilde{S}_0 exp\big( (\mu − r − {\frac{1}{2}}{\sigma}^2) t + {\sigma} B_t \big)$$
$$log(\tilde{S}_t / \tilde{S}_0) =  (\mu − r − {\frac{1}{2}}{\sigma}^2) t + {\sigma} B_t $$
So, the log prices is exactly Brownian motion with a drift. Stock prices are not Levy but Log Stock prices are, hence the `esscher_space="log"`. So we don’t need a new simulation method. We reuse `_gauss_paths` and exponentiate. That is the whole of `stocks.simulate`.

In [30]:
def __init__(self, mu, sigma, r, seed=None, rho=0.0, verbose=True):
    super().__init__(seed)
    self.mu, self.sigma, self.r, self.rho = _pair(mu, "mu"), _pair(sigma, "sigma"), float(r), float(rho)
    self.A = _vol(self.sigma, self.rho)
    self.Sigma = self.A @ self.A.T
    self.independent = self.rho == 0.0
    self._desc = ...

@property
def log_drift(self):
    return self.mu - self.r - 0.5 * np.diag(self.Sigma)
    
def simulate(self, n, times, start, rng):
    return start * np.exp(_gauss_paths(n, times, self.log_drift, self.A, rng))

def marginal(self, t, start, k):
    return stats.lognorm(np.sqrt(self.Sigma[k, k] * t), scale=start[k] * np.exp(self.log_drift[k] * t))

The `__init__` for `stocks` is almost exactly same as `bm`'s `__init__`. The small technical changes are that `mu`, `sigma` and `r` have no defaults, so you must give all three, for example `stocks(0.08, 0.2, 0.03)`.

Here `log_drift` is the number $\mu−r−{\frac{1}{2}}{\sigma}^2$ computed in the class. `np.diag(self.Sigma)` pulls out the diagonal of the covariance matrix. `@property` lets you write `self.log_drift` without parentheses, as if it were a stored value. It is recomputed from `mu`, `r` and `Sigma` each time you access it. This is why it’s a property and not stored in `__init__` as it is always consistent with the current parameters.

Now for the `pdf` we already saw the log-stock prices are 2D Brownian motion with a drift, so they will have `_bvn_pdf` distribution that we have already built. The change of variable rule is
$$f_{\text{price}}(x) = f_{\text{log}}(\text{log} x) · (1/x)$$
because $u=log(x) \implies du = dx/x$. So, the factor `1/x` is the stretch correction we need due to `log`. In 2D, 
$$f_{\text{price}}(x, y) = f_{\text{log}}(log x, log y) · 1/(x · y)$$
So we will have the following line in the codes
```text
p = _bvn_pdf(lx, ly, self.log_drift * t, self.Sigma * t) / (x * y)
```

In [31]:
def pdf(self, t, x, y, start=None):
    s = self._start(start)
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = (x > 0) & (y > 0)
    lx, ly = np.log(np.where(ok, x, 1.0) / s[0]), np.log(np.where(ok, y, 1.0) / s[1])
    p = _bvn_pdf(lx, ly, self.log_drift * t, self.Sigma * t) / (np.where(ok, x, 1.0) * np.where(ok, y, 1.0))
    return np.where(ok, p, 0.0)

`lx`, `ly` are the the points, converted to the log scale. We divide by the start first, $log(x / s[0])$, because the Brownian part is the log of the ratio to the starting price. The `ok` part are just checks the valid points and `where` ensures ONLY valid `x` and `y` go through. `where(ok, x, 1.0)` means "use x where valid, or else use 1.0".

## 2.1.5 Change of measure

Let us discuss the architecture behind what we are going to do next. This particular section was the main reason I started making this library to begin with. There was no easy way to see the abstract notion of "change of measure" geometrically in Python. These fall under the theory of *Radon-Nikodym Derivatives*. Let us forget RN derivates for now. Let us try to make our own method for removing drift.

For certain "nice" stochastic process families (exponential families / Lévy processes like Brownian motion, geometric Brownian motion, Poisson, and Gamma), applying an Esscher tilt or Girsanov measure change does not create a bizarre, unknown distribution. Instead, the tilted process belongs to the exact same process class, just with shifted parameters:
1. Tilting a `bm(mu, sigma)` produces another `bm` with drift $\mu + \Sigma \theta$.
2. Tilting a `poisson(rate)` produces another `poisson` with rate $\lambda e^\theta$.
3. Tilting a `gamma_process(a, b)` produces another `gamma_process` with rate $b - \theta$.
4. Tilting or changing numéraire on `stocks(mu, sigma, r)` produces another `stocks` object with a new drift. (yet to see this).

Because the tilted process is still a genuine `Process` instance of the exact same type, we already know its exact analytical `.pdf()`, `.center()`, and `.simulate()` formulas. We do not need to approximate anything with Monte Carlo weights. We just construct that new `Process` object and call `.pdf()` directly. That is what "closed-form" will mean to the viewer and note that there is nothing abstract about this. There is going to be another method called "Reweight", which we will see later, and that is going to be the "abstract counter-part" of this closed-form method.

We have seen the tilting mechanism already for `bm` explicitly. For stocks it works the same way, with one change in where the tilt acts on `log S`, not on `S`. That is what `esscher_space = "log"` will mean. `log S` is a Brownian motion with drift, and that is the thing we know how to tilt already. The only technicality is for `bm`, tilting bu $\theta$ shifted the drift by $\Sigma \theta$, which will happen here too but we do want to return to the Stocks. That is what `_clone` will do.

In [32]:
def _clone(self, log_drift):
    mu = log_drift + self.r + 0.5 * np.diag(self.Sigma)
    return stocks(mu, self.sigma, self.r, self.seed, self.rho, verbose=False)

def esscher(self, theta):
    return self._clone(self.log_drift + self.Sigma @ np.asarray(theta, float))

Now we will include another special feature to the stock prices.

This is also going to be related to drift change similar to what $\theta$-tilt does. These are 4 special types of drifts. $\theta$ slider only gave us the ability to move along $\begin{bmatrix}1 \\ 1\end{bmatrix}$.

So far, `P` has been the real-world law, that is, the probabilities under which the stock actually has drift $\mu$. A measure is just a choice of probabilities for the same random paths. Changing the measure keeps the same paths but reweights how likely each one is (we’ll see exactly how in the next part, weights).

This file will offer four measures, listed in the class attribute measures = `("P", "Q", "Q1", "Q2")`, which is what the viewer’s toggle reads:

`P`: the real world, with drift μ.
`Q`: the risk-neutral world. The probabilities are chosen so the discounted prices have zero drift (a fair game). This is the world used for pricing derivatives.
`Q1`: the world where stock 1 is the unit of account (the “numeraire”). Prices are measured in units of stock 1, and not in dollars.
`Q2`: the same, with stock 2 as the unit.
The key fact that makes this easy

In this model, switching measure only changes the drift. The volatility `A` and the correlation stay the same (this is Girsanov’s theorem, which we only need as a fact). So under each measure, `log S` is still a Brownian motion with the same `Σ`, just with a different drift, and “the process under `Q`” is again a `stocks` object, with a different `mu`. That is why we can return a ready-made object for each measure. The following code returns the drift of `log S` under the chosen measure.

In [33]:
def _drift_under(self, measure):
    half = 0.5 * np.diag(self.Sigma)
    if measure == "P":
        return self.log_drift
    if measure == "Q":
        return -half
    if measure in ("Q1", "Q2"):
        return self.Sigma[:, int(measure[1]) - 1] - half
    raise ValueError(f"unknown measure {measure!r}")

$\bullet$ P: the usual `log_drift = μ − r − ½σ²`.

$\bullet$ Q: the discounted price must have no drift, so the `μ − r` part is deleted and only the correction `−½σ²` remains. Substituting into `_clone`’s formula, `mu = log_drift + r + ½σ² = r`. So under Q, the stock’s drift is exactly the interest rate `r`, which is what “risk-neutral” means.

$\bullet$ Q1 / Q2: the drift is `Σ[:, j] − ½σ²`, where j is the stock used as the unit. `measure[1]` is the character after the Q, so `"Q1"` gives `"1"`, `int(...)` makes it 1, and the `− 1` turns it into the column index 0. The column `Σ[:, j]` holds the covariances of each stock with stock j. Measuring in units of stock j adds each stock’s covariance with stock j to its drift, because the unit itself is moving, and the stocks that move together with it look less volatile in its units. We won’t derive this here.

$\bullet$ Anything else raises an error.

In the base `Process` class we have `at` that only handles `measure=P` and delegates directly to `self.esscher([theta, theta])`. We did not discuss this earlier but we had this in base class `Process`:

In [34]:
def at(self, theta=0.0, measure="P"):
        """Closed-form variant of the process under an Esscher tilt / alternative measure."""
        if theta == 0 and measure == "P":
            return self
        if self.esscher_space is None or measure != "P":
            raise NotImplementedError(f"{self!r} does not support tilting / measure change")
        return self.esscher(np.array([theta, theta], float))

In `stocks` we need to override this with a new `at` because here both discrete financial measures (`P`, `Q`, `Q1`, `Q2`) and continuous Esscher tilting are supported.

Here, `at` will be doing this: “give me the process under this measure, tilted by this θ.” It combines the two operations.
1. Do nothing if `θ = 0` and the measure is `P`: return `self`.
2. Otherwise take the drift under the measure, add the Esscher tilt `Σθ`, and let `_clone` convert that `log_drift` back into a `mu` and build the new object.

This overrides the base class’s `at`, which handled tilts only. The base version raised an error for any measure other than `P`. Stocks can support the others because of the key fact above, so it replaces at with a version that accepts all four.

In [35]:
def at(self, theta=0.0, measure="P"):
    if theta == 0 and measure == "P":
        return self
    return self._clone(self._drift_under(measure) + self.Sigma @ np.array([theta, theta], float))

So far `at` returns a `stocks` with a different `mu`. This is the closed-form route and now we see a 2nd route. We will call this 2nd method `"reweight"`.
1. Simulate under the original measure `P`, as usual.
2. We won’t touch the paths. Instead give each path a weight saying how much more (or less) likely it is in the new world. When we get to actual plots this will be seen as "the paths staying same but colours changing". *THIS IS LITERALLY WHAT RADON-NIKODYM SAYS, THE ELEMENTS THEMSELVES DO NOT CHANGE BUT ARE REWEIGHTED*.

Understand that visually they will do exactly the same thing: "The paths won't change at all, but the colours will change" and that is kinda the point. It establishes that going via both route gives same output. Closed-form is what someone will do if they dont know what this abstract idea of "Radon-Nikodym" is. 

closed-form: Calls `m.at(theta, measure).pdf(...)`. Because tilting a Gaussian/lognormal process just shifts its drift parameter analytically (`_clone`), we plug the new drift directly into the exact bivariate lognormal formula.

reweighting (dQ/dP): Ignores the analytical formula. It takes the 20,000 raw samples `self.big` simulated under $P$, multiplies each sample by its Radon–Nikodym weight $w = \frac{dQ}{dP}$ via `m.weights()`.

When we actually get to the plotting layer (we haven't yet) of this library, we can literally distinguish between the two density functions. The one generated using "closed-form" is nicer looking and smooth since it is done via re-evaluating the `pdf` whereas the one done via `reweight` will looks rough and built with lots of troughs and grooves (this will look like an approximation of the surface built via "closed-form").

In [36]:
def weights(self, t, X, start, theta=0.0, measure="P"):
    w = np.ones(len(X))
    if theta != 0 and self.esscher_space:
        th = np.array([theta, theta], float)
        w = w * np.exp(self.levy_state(X, start) @ th - t * self.cumulant(th))
    return w

We will add this above code for method `weights` in base class `Process` first so that every sub-class and and sub-sub-class inherits them. Obviously overwriting is allowed (we already did that with `at`). The tilted law is $q(x) = w(x) p(x)$ and  $$w(x)\,=\,\frac{e^{\theta x}}{E[e^{\theta X}]}=\exp(\theta \cdot Y_t(\omega) - t\kappa(\theta))$$
We have the seen the theory already in **chapter 01** where we explain why this is the formulae for weight.
```text
w = exp( θ · (state) − t · κ(θ) )
```
where “state” is `X_t − start` for the `raw` processes (and `log(X_t/start`) for `stocks`). Let us examine the base class's `weights`'s codes now.

   1. Inputs: `X` is the cloud at time `t`, shape `(N, 2)`. The output is one weight per sample, shape `(N,)`.
   2. `np.ones(len(X))`: the default is “every sample counts 1”, meaning no change (paths being multiplied by 1 does no reweighting).
   3. The `if`: the tilt is applied only when $\theta \neq 0$ and the process allows tilting (`esscher_space` is set, as in the earlier explanation of that flag).
   4. `th = [theta, theta]`: the single slider, copied into both coordinates.
   5. `levy_state(X, start) @ th`: each sample’s state row dotted with th, which gives θx·x + θy·y for every sample at once because "`levy_state(X, start)`" is `X_t − start` for the `raw` processes (and `log(X_t/start`) for `stocks`). This is the `θ·(state)` part.
   6. `t * self.cumulant(th)` is the `t·κ(θ)` part.

It works for `bm`, `poisson` and `gamma_process`, which have a `cumulant`. As a check, tilting a Gaussian this way moves the mean by $\Sigma \theta \cdot t$ (explicitly calculated in chapter 01, section 2), which matches `bm.esscher`.

Why we need an improvement? The base version has two limits, and `stocks` hit both.

1. It ignores measure completely. The argument is accepted (notice we have `measure="P"`) and never read. For a `stock`, `weights(..., measure="Q")` with `θ = 0` would skip the `if` and return all `ones`, silently claiming “Q looks exactly like P”. The code gives no error and the answer is wrong, because `Q` has a different drift. We want to fix all that and want to ensure we get different drifts for each of `P`, `Q`, `Q1`, `Q2`.

2. A tilt can only move the drift in a limited way. A tilt changes the drift by $\Sigma\, \begin{bmatrix}\theta \\ \theta\end{bmatrix}$ (keep in mind, we as creator put this restriction even though the underlying math can take vector valued `theta` because this is good enough to convey the ideas), one number along a fixed direction (the single-slider limit). A measure change needs the drift to move to a specific target. For `Q`, the target is a price drift of exactly `r`, so the change is `−(μ − r)`. That is generally not of the form `Σ·(θ,θ)`, so no single tilt can produce it.

So for `stocks` we need a weight formula that handles any move from one drift to another, not just “tilt along `Σ(1,1)`”. Both limits point to the same fix of finding a weight formula built from the two drifts (the starting one and the target one) instead of from $\theta$ and $\kappa$. We will call it `stock.weights`.

The key observation

For stocks, every world we care about has the same shape and `state`: `log S_t` is Gaussian with covariance `Σt`, and only the drift changes. So the weight q/p is just the ratio of two Gaussian densities with the same spread but different centers. We don’t need $\kappa$ or $\theta$ at all, only the two drifts. The code calls them:

`m_from`: the drift we sampled under (`log_drift` under P)

`m_to`: the drift of the world we want

We have already seen the math behind what we are going to code now. It was discussed in **chapter 01, section 2**. We are going to use the *Radon-Nikodym* and *Girsanov's form*:
$$Z_t = \frac{d\mathbb{Q}}{d\mathbb{P}}\Big\vert{}_{\mathcal{F}_t} = \exp\left(-W_t \cdot \vartheta - \frac{1}{2}\Vert{}\vartheta\Vert{}^2 t\right)$$
We know this is equivalent to the *Esscher's form*.

In [37]:
def weights(self, t, X, start, theta=0.0, measure="P"):
    if measure == "P" and theta == 0:
        return np.ones(len(X))
    m_from = self.log_drift
    m_to = self._drift_under(measure) + self.Sigma @ np.array([theta, theta], float)
    Y = np.log(X / start)
    W = np.linalg.solve(self.A, (Y - m_from * t).T).T
    th = np.linalg.solve(self.A, m_from - m_to)
    return np.exp(-W @ th - 0.5 * th @ th * t)

We will not discuss the `stocks.exchange` here. That is pretty much done in the math-discussion part itself. The math is the hard part. `stocks_neutral` is just `stocks` under `Q` as its own class. Everything inherited from stocks (simulate, marginal, pdf) now runs in the risk-neutral world automatically, because the object believes its price drift is `r`.

The real `mu` is stored separately as `mu_real`, and it is used only for display. It has no effect on any number the class produces. Only `r`, `sigma` and `rho` do. Infact if you write
```python
m = rd.stocks_neutral(0.08, 0.2, 0.03)    # mu, sigma, r
```
we will get the output that shows this is using a different Brownian motion $\{B^*_t\}$.

In [38]:
import radonika as rd

m = rd.stocks_neutral(0.08, 0.2, 0.03)    # mu, sigma, r

**Discounted stock price SDE with Brownian fluctuation B^*_t = B_t + ((μ-r)/ σ) t:**

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

The last few things are the viewer wrappers namely `plot_law`, `tilt`, `analyze` and `rare`. They are the main visual tools which we will make in next chapter. 

```text
Process (base)        simulate [empty] · marginal [empty]
 │                    _rng · _start · paths · sample_slices · sample
 │                    pdf (product rule) · center
 │                    at · weights (Esscher only) · plot_law/tilt/analyze/rare
 │
 ├─ _IID ───────────  simulate · marginal · logpdf
 │    └─ normal, cauchy, exponential, uniform, student_t, laplace, lognormal
 │
 ├─ bm ─────────────  simulate (steps added up) · marginal · pdf (_bvn_pdf)
 │                    cumulant · esscher · theta_star
 ├─ bm_bridge ──────  simulate (loop) · marginal (arch-shaped variance)
 ├─ poisson, gamma_process, cauchy_process
 │
 └─ stocks ─────────  exp of Brownian · pdf (log scale ÷ xy)
      │               levy_state · cumulant · esscher/_clone · theta_star
      │               at (tilt + measure) · weights (any drift → any drift)
      │               exchange_option
      └─ stocks_neutral   (mu replaced by r)

helpers: _pair · _vol · _bvn_pdf · _gauss_paths
```

This concludes the `process.py` file.

<p align="center">
  <strong>--------------- chapter 02 Section 01 End ---------------</strong>

</p>